# 1장 2강: 유의수준·검정력·표본 크기의 트레이드오프와 실질적 유의성 — 실습문제

## 실습 목표

- 효과 크기와 유의수준이 같을 때 표본 크기에 따라 검정력이 어떻게 변하는지 설명할 수 있다.
- 목표 검정력과 유의수준을 이용해 필요한 표본 크기를 계산할 수 있다.
- 두 집단의 평균 차이에 대한 Cohen’s d를 계산하고 해석할 수 있다.
- p-value와 효과 크기, 실무 기준을 함께 사용하여 의사결정 근거를 작성할 수 있다.

## 실습 환경 / 데이터

- Python
- NumPy, pandas
- scipy.stats
- statsmodels
- `ames_housing.csv`

주요 컬럼은 다음과 같습니다.

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `CentralAir` | 중앙 냉방시설 유무(`Y`, `N`) |
| `KitchenQual` | 주방 품질(`Ex`, `Gd`, `TA`, `Fa`) |

> 모든 검정은 별도 지시가 없으면 유의수준 `α = 0.05`를 사용합니다.  
> Cohen’s d는 절댓값을 기준으로 약 0.2는 작은 효과, 0.5는 중간 효과, 0.8 이상은 큰 효과로 해석합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 전체 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [2]:
# 실습 준비 코드를 작성하세요.
# 1. 필요한 라이브러리
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Ames Housing 데이터 불러오기
df = pd.read_csv(r'C:\Machine_Learning_worksheet\Data\ames_housing.csv', encoding= 'utf-8-sig')   # 실습 파일명에 맞게 수정

# 3. 데이터 확인
print('데이터 크기:', df.shape)
print('전체 결측치 수:', df.isnull().sum().sum())
print('컬럼명:', df.columns.tolist())

df.head()

데이터 크기: (1460, 10)
전체 결측치 수: 0
컬럼명: ['SalePrice', 'GrLivArea', 'LotArea', 'OverallQual', 'KitchenQual', 'CentralAir', 'HeatingQC', 'PavedDrive', 'Neighborhood', 'YearBuilt']


,SalePrice,GrLivArea,LotArea,OverallQual,KitchenQual,CentralAir,HeatingQC,PavedDrive,Neighborhood,YearBuilt
0,208500,1710,8450,7,Gd,Y,Ex,Y,CollgCr,2003
1,181500,1262,9600,6,TA,Y,Ex,Y,Veenker,1976
2,223500,1786,11250,7,Gd,Y,Ex,Y,CollgCr,2001
3,140000,1717,9550,7,Gd,Y,Gd,Y,Crawfor,1915
4,250000,2198,14260,8,Gd,Y,Ex,Y,NoRidge,2000


---

## 필수 1. 표본 크기와 검정력 비교

### 문제 1-1. 중앙 냉방시설 비교 연구에 필요한 표본 수 설계

#### 문제 설명

중앙 냉방시설이 있는 주택과 없는 주택의 판매가격을 비교하는 연구를 준비하고 있습니다. 사전 조사에서 표준화 효과 크기는 `0.3` 정도로 예상했습니다.

다음 조건에서 그룹당 표본 크기에 따른 검정력을 비교하고, 목표 검정력 0.8을 확보하기 위해 필요한 표본 수를 계산하세요.

- 예상 효과 크기: `0.3`
- 유의수준: `0.05`
- 양측검정
- 비교할 그룹당 표본 수: `30명`, `100명`, `300명`

#### 요구사항

1. `TTestIndPower()` 객체를 생성하세요.
2. 그룹당 표본 수가 30명, 100명, 300명일 때의 검정력을 각각 계산하세요.
3. 표본 크기별 검정력을 소수점 셋째 자리까지 출력하세요.
4. 유의수준 0.05, 목표 검정력 0.8, 효과 크기 0.3일 때 그룹당 필요한 표본 수를 계산하세요.
5. 계산된 표본 수는 소수점 이하를 올림하여 정수로 출력하세요.
6. 세 표본 크기 중 목표 검정력 0.8을 충족하는 경우를 확인하세요.

#### 해석 질문

**Q1.** 효과 크기와 유의수준이 같을 때 표본 크기가 커지면 검정력은 어떻게 변하나요?  
**Q2.** 검정력이 낮으면 실제 차이가 존재할 때 어떤 오류의 위험이 커지나요?  
**Q3.** 필요한 표본 수를 소수점 이하 올림으로 처리하는 이유는 무엇인가요?  
**Q4.** 표본 수를 무조건 크게 설정하는 것이 항상 최선인가요?

#### 제출 결과

- 표본 크기별 검정력
- 목표 검정력에 필요한 그룹당 표본 수
- 목표 검정력 충족 여부
- Q1~Q4 답변

In [3]:
# 필수 1 코드를 작성하세요.
import math
from statsmodels.stats.power import TTestIndPower

# 1. 객체 생성
analysis = TTestIndPower()

effect_size = 0.3
alpha = 0.05
target_power = 0.8

# 2~3. 표본 크기별 검정력 계산 (소수점 셋째 자리)
for n in [30, 100, 300]:
    power = analysis.power(effect_size=effect_size, nobs1=n,
                           alpha=alpha, ratio=1.0, alternative='two-sided')
    print(f"그룹당 {n}명 → 검정력: {power:.3f}")

# 4~5. 목표 검정력 0.8을 위한 그룹당 필요 표본 수 (올림)
required_n = analysis.solve_power(effect_size=effect_size, power=target_power,
                                  alpha=alpha, ratio=1.0, alternative='two-sided')
print(f"\n필요 표본 수(그룹당): {math.ceil(required_n)}명")

# 6. 목표 검정력 충족 여부
print()
for n in [30, 100, 300]:
    power = analysis.power(effect_size=effect_size, nobs1=n,
                           alpha=alpha, ratio=1.0, alternative='two-sided')
    print(f"{n}명: {'충족' if power >= target_power else '미충족'} (검정력 {power:.3f})")

그룹당 30명 → 검정력: 0.208
그룹당 100명 → 검정력: 0.560
그룹당 300명 → 검정력: 0.956

필요 표본 수(그룹당): 176명

30명: 미충족 (검정력 0.208)
100명: 미충족 (검정력 0.560)
300명: 충족 (검정력 0.956)


### 필수 1 답변 작성란

- **Q1.** 표본 크기가 커지면서 검정력이 높아진다. 표본이 커질수록 표준오차가 작아져 포본분포가 좁아지고, 같은 킉의 자이라도 우연으로 설명하기 어려워지기 때문. 다만 증가속도는 일정하지 않음. 수익 체감 구조 올라갈 수록
- **Q2.**
2종 오류의 위험이 커짐. 검정력은 1-beta 이므로 검정력 0.028이면 beta= 0.792, 즉 실제로 차이가 존재하는데도 귀무가설을 기각하지 못할 수 있음. 그룹당 30명 설계로 차이엇ㅂ음이라는 결론이 난다면 그 결론은 거의 정보가 없다고 보는게 맞다
- **Q3.**
포본 수는 사람, 개체 단위라 정수여야 하고, 내림하면 목표 거멎ㅇ력에 미달하기 때문. 175.385명에서 175명으로 내리면 검정력이 0.8밑으로 떨어짐 요구조건을 최소한 충족 시키려면 올림이 유일하게 안전한 방법

- **Q4.**
아니요
비용: 표본 수직 시에는 시간, 비용 그리고 참여자 부담이 따르고, 앞서 본 수익 체감 때문에 어느지점부터는 추가 표본의 가치가 급격히 떨어짐
실질적 유의성: 표본이 충분히 크면 실무적으로 무의미할 만큼 작은 차이도 통계적으로 유의미 해진다. p값은 작은 효과 크기는 무시할 수준인 상황이 생김
윤리:임상시험처럼 위험이 따르는 연구에서는 필요 이상으로 많은 대상자를 노출시키는 것 자체가 문제입니다.

---

## 필수 2. 통계적 유의성과 실질적 유의성 종합 판단

### 문제 2-1. 중앙 냉방시설 유무에 따른 판매가격 차이 분석

#### 문제 설명

한 부동산 회사는 중앙 냉방시설이 있는 주택과 없는 주택의 평균 판매가격 차이를 분석하려고 합니다. 회사는 두 집단의 평균 판매가격 차이가 **100,000달러 이상**이어야 냉방시설 설치 지원 사업을 검토할 실무적 가치가 있다고 정했습니다.

#### 요구사항

1. `CentralAir == "Y"`인 주택의 `SalePrice`를 `air_yes`에 저장하세요.
2. `CentralAir == "N"`인 주택의 `SalePrice`를 `air_no`에 저장하세요.
3. 두 집단의 표본 수와 평균 판매가격을 출력하세요.
4. `stats.ttest_ind()`로 두 집단 평균 차이에 대한 양측검정을 수행하세요.
5. 제공된 공식에 따라 Cohen’s d 계산 함수를 작성하고 효과 크기를 구하세요.
6. 평균 차이, p-value, Cohen’s d를 출력하세요.
7. 다음 기준으로 결과를 판단하세요.
   - `p-value ≤ 0.05`: 통계적으로 유의함
   - `|Cohen’s d| ≥ 0.8`: 큰 효과
   - `|평균 차이| ≥ 100000`: 회사 기준에서 실질적으로 유의함
8. 세 결과를 종합하여 사업 검토 여부에 대한 근거를 작성하세요.

#### Cohen’s d 공식

두 집단의 평균 차이를 합동 표준편차로 나누어 계산합니다.

$$
d = \frac{\bar{x}_1 - \bar{x}_2}{s_p}
$$

$$
s_p = \sqrt{\frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1+n_2-2}}
$$

- $\bar{x}_1$, $\bar{x}_2$: 각 집단의 평균
- $s_1$, $s_2$: 각 집단의 표본 표준편차
- $n_1$, $n_2$: 각 집단의 표본 수
- $s_p$: 합동 표준편차

#### 해석 질문

**Q1.** p-value와 Cohen’s d는 각각 어떤 정보를 제공하나요?  
**Q2.** 두 집단의 판매가격 차이는 통계적으로 유의한가요?  
**Q3.** Cohen’s d를 기준으로 효과 크기는 어느 정도인가요?  
**Q4.** 회사가 정한 100,000달러 기준에서 실질적으로 유의한가요?  
**Q5.** 통계적으로 유의하고 효과 크기가 크더라도 회사의 실무 기준을 충족하지 못할 수 있나요?

#### 제출 결과

- 집단별 표본 수와 평균
- 독립표본 t검정 결과
- 평균 차이와 Cohen’s d
- 통계적·효과크기·회사 기준 판단
- 최종 의사결정 근거
- Q1~Q5 답변

In [7]:
# 필수 2 코드를 작성하세요.
import numpy as np
import pandas as pd
from scipy import stats

GROUP_COL = 'CentralAir'    # 공백 없음
VALUE_COL = 'SalePrice'
THRESHOLD = 100_000         # 회사 실무 기준

g = df[[GROUP_COL, VALUE_COL]].dropna()
labels = sorted(g[GROUP_COL].unique())
a = g.loc[g[GROUP_COL] == labels[0], VALUE_COL]
b = g.loc[g[GROUP_COL] == labels[1], VALUE_COL]

# 1) 집단별 표본 수와 평균
print(g.groupby(GROUP_COL)[VALUE_COL].agg(n='count', mean='mean', std='std').round(2))

# 2) 독립표본 t검정 (등분산 가정 없이 Welch)
t_stat, p_val = stats.ttest_ind(a, b, equal_var=False)
print(f"\nt = {t_stat:.3f}, p-value = {p_val:.4g}")

# 3) 평균 차이와 Cohen's d (pooled SD)
diff = b.mean() - a.mean()
n1, n2 = len(a), len(b)
pooled_sd = np.sqrt(((n1-1)*a.var(ddof=1) + (n2-1)*b.var(ddof=1)) / (n1+n2-2))
d = diff / pooled_sd
print(f"평균 차이 = {diff:,.2f}달러, Cohen's d = {d:.3f}")

# 평균 차이의 95% 신뢰구간 (실무 기준 판단에 필요)
se = np.sqrt(a.var(ddof=1)/n1 + b.var(ddof=1)/n2)
dfree = se**4 / ((a.var(ddof=1)/n1)**2/(n1-1) + (b.var(ddof=1)/n2)**2/(n2-1))
lo, hi = diff + np.array([-1, 1]) * stats.t.ppf(0.975, dfree) * se
print(f"차이의 95% CI = [{lo:,.2f}, {hi:,.2f}]")

# 4) 세 가지 기준 판단
print(f"\n통계적 유의성: {'유의함' if p_val < 0.05 else '유의하지 않음'} (α=0.05)")
size = '큼' if abs(d) >= 0.8 else '중간' if abs(d) >= 0.5 else '작음' if abs(d) >= 0.2 else '미미함'
print(f"효과 크기: {size} (|d|={abs(d):.3f})")
print(f"회사 기준({THRESHOLD:,}달러): {'충족' if abs(diff) >= THRESHOLD else '미충족'}")

               n       mean       std
CentralAir                           
N             95  105264.07  40671.27
Y           1365  186186.71  78805.21

t = -17.268, p-value = 2.281e-37
평균 차이 = 80,922.64달러, Cohen's d = 1.052
차이의 95% CI = [71,662.29, 90,182.99]

통계적 유의성: 유의함 (α=0.05)
효과 크기: 큼 (|d|=1.052)
회사 기준(100,000달러): 미충족


### 필수 2 답변 작성란

- **Q1.**
p-value 는 관측된 차이가 통계적으로 유의한지 판단하는 데 사용
- Cohen's d는 두 집단 차이의 크기를 표준화 하여 보여줌

- **Q2.**
- 유의 함 p-value 가 0.05보다 작음
-귀무가설: 냉방 시설 유무는 주택 가격 평균에 차이를 보이지 않는다 -> 기각

- **Q3.**
- Cohen's의 절대값이 1.05고 규정한 0.8보다 크기 때문에 큰 효과로 해석 할 수 있다.

- **Q4.**
- 유의하지 않음, 80,900달러 정도 였기 때문에 회사의 기대 미충족


- **Q5.**
그렇다
통계적 유의성은 차이가 있다는 근거가 있는가?
효과 크기는 자료의 흩어짐에 비해 차이가 큰가?
회사 입장은 업무에서 요구하는 최소 금액 차이를 넘는가?



---

## 과제. 주방 품질에 따른 판매가격 차이 분석

### 문제 3-1. 주방 품질이 `Gd`인 집단과 `TA`인 집단 비교

#### 문제 설명

부동산 중개회사는 주방 품질이 `Gd`(Good)인 주택과 `TA`(Typical/Average)인 주택의 평균 판매가격을 비교하려고 합니다. 회사는 평균 판매가격 차이가 **50,000달러 이상**이면 마케팅에서 주방 품질의 차이를 강조할 실무적 가치가 있다고 판단합니다.

> 필수 2에서 학습한 검정과 효과 크기 계산 절차를 새로운 두 집단에 적용하는 과제입니다.

#### 요구사항

1. 주방 품질이 `Gd`인 집단과 `TA`인 집단의 `SalePrice`를 각각 준비하세요.
2. 두 집단의 표본 수와 평균 판매가격을 출력하세요.
3. 두 집단의 평균 차이에 대한 양측 독립표본 t검정을 수행하세요.
4. Cohen’s d를 계산하세요.
5. 평균 차이, p-value, Cohen’s d를 출력하세요.
6. 다음 기준에 따라 각각 판단하세요.
   - 통계적 유의성: `p-value ≤ 0.05`
   - 큰 효과: `|Cohen’s d| ≥ 0.8`
   - 실무적 유의성: `|평균 차이| ≥ 50000`
7. 세 가지 판단을 종합하여 주방 품질을 마케팅에서 강조할 근거가 있는지 결론을 작성하세요.

#### 해석 질문

**Q1.** 두 집단의 판매가격 차이는 통계적으로 유의한가요?  
**Q2.** Cohen’s d를 기준으로 효과 크기는 어느 정도인가요?  
**Q3.** 회사가 정한 50,000달러 기준을 충족하나요?  
**Q4.** 최종적으로 주방 품질 차이를 마케팅에서 강조할 근거가 있다고 볼 수 있나요?

#### 제출 결과

- 집단별 표본 수와 평균
- t통계량과 p-value
- 평균 차이와 Cohen’s d
- 통계적·실질적 유의성 판단
- 최종 결론
- Q1~Q4 답변

In [3]:
# 과제 코드를 작성하세요.


import numpy as np
from scipy import stats

# 1) 두 집단 준비
gd = df.loc[df['KitchenQual'] == 'Gd', 'SalePrice'].dropna()
ta = df.loc[df['KitchenQual'] == 'TA', 'SalePrice'].dropna()

# 2) 표본 수와 평균
n1, n2 = len(gd), len(ta)
m1, m2 = gd.mean(), ta.mean()
print(f"Gd: n = {n1}, 평균 = {m1:.4f}")
print(f"TA: n = {n2}, 평균 = {m2:.4f}")

# 3) 양측 독립표본 t검정 (Welch)
t_stat, p_value = stats.ttest_ind(gd, ta, equal_var=False)

# 4) Cohen's d (합동표준편차)
s1, s2 = gd.std(ddof=1), ta.std(ddof=1)
s_pooled = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))
d = (m1 - m2) / s_pooled

# 5) 출력
diff = m1 - m2
print(f"\n평균 차이   = {diff:.4f}")
print(f"t-통계량    = {t_stat:.4f}")
print(f"p-value     = {p_value:.4f}  (지수: {p_value:.4e})")
print(f"Cohen's d   = {d:.4f}")

# 6) 세 기준 판단
print(f"\n통계적 유의성: p={p_value:.4f} → {'유의함' if p_value <= 0.05 else '유의하지 않음'}")
print(f"큰 효과      : |d|={abs(d):.4f} → {'큰 효과' if abs(d) >= 0.8 else '큰 효과 아님'}")
print(f"실무적 유의성: |차이|={abs(diff):,.2f} → {'기준 충족' if abs(diff) >= 50000 else '기준 미달'}")

Gd: n = 586, 평균 = 212116.0239
TA: n = 735, 평균 = 139962.5116

평균 차이   = 72153.5123
t-통계량    = 23.9812
p-value     = 0.0000  (지수: 4.9330e-99)
Cohen's d   = 1.3991

통계적 유의성: p=0.0000 → 유의함
큰 효과      : |d|=1.3991 → 큰 효과
실무적 유의성: |차이|=72,153.51 → 기준 충족


### 과제 답변 작성란

**Q1.** 두 집단의 판매가격 차이는 통계적으로 유의한가요? 
 p value가 0.05 보다 낮기 때문에 유의 함

**Q2.** Cohen’s d를 기준으로 효과 크기는 어느 정도인가요?
Cohen's d기준으로 는 큰 효과이다. 두 표준 편차에 1.4배인거 는 통계적으로 큰 숫자이기 때문에

**Q3.** 회사가 정한 50,000달러 기준을 충족하나요?  
충족합니다. 평균차이가 72,153.51 달러로 회사가 정한 50,000달러보다 22,153.51달러 많다.


**Q4.** 최종적으로 주방 품질 차이를 마케팅에서 강조할 근거가 있다고 볼 수 있나요?
있습니다. 
P-value 는 0.05 이하이므로 유의 기준 밑이고 
회사에서 정한 5만달러 보다 이상이고 효과 크기도 1.4배이다.
이에 따라, 주방품질의 차이를 마케팅에서 강조할 근거가 충분하다.



---

## 실습 마무리

1. 검정력은 무엇이며 제2종 오류와 어떤 관계가 있나요?
2. 같은 효과 크기와 유의수준에서 표본 크기가 커지면 검정력은 어떻게 변하나요?
3. 목표 검정력이 높거나 발견하려는 효과가 작을수록 필요한 표본 수는 어떻게 변하나요?
4. p-value와 Cohen’s d를 함께 확인해야 하는 이유는 무엇인가요?
5. 통계적으로 유의한 결과가 반드시 실무적으로 중요한 결과를 의미하나요?